# Bounded T1 XAI and Synthetic T2 Diagnostics

This notebook runs real, reproducible occlusion experiments using the recovered T1 adapter. T1 image explanations use tile occlusion, not Grad-CAM. T2 outputs and attributions are synthetic auxiliary diagnostics only and are not veterinary or clinical evidence.

In [ ]:
%pip install -q -U "transformers>=4.50.0" peft accelerate bitsandbytes pillow scikit-learn safetensors shap
import torch
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Select an L4 or T4 GPU runtime before continuing")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
from pathlib import Path
import getpass
import os
import tarfile
import zipfile

uploaded = files.upload()
bundle_name = next(name for name in uploaded if name.endswith(".tar.gz"))
bundle_path = Path("/content") / bundle_name
root = Path("/content/xai_t2_bundle")
root.mkdir(exist_ok=True)
with tarfile.open(bundle_path, "r:gz") as archive:
    archive.extractall(root)
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Paste HF_TOKEN: ")
with zipfile.ZipFile(root / "medgemma-t1-qlora-adapter.zip") as archive:
    archive.extractall(root)
adapter = root / "medgemma-t1-qlora"
results = Path("/content/xai_t2_results")
results.mkdir(exist_ok=True)
assert (adapter / "adapter_model.safetensors").exists()
print("Bundle:", root)
print("Adapter:", adapter)

In [ ]:
import subprocess
import sys

runner = root / "scripts"
subprocess.run([
    sys.executable, "-u", str(runner / "attribute_t1_occlusion.py"),
    "--root", str(root),
    "--input", str(root / "data/secondary/instruction_tuning_mixed/val.jsonl"),
    "--adapter", str(adapter),
    "--manifest", str(root / "data/secondary/manifests/images_prepared.jsonl"),
    "--output", str(results / "t1_image_attributions.jsonl"),
    "--max-examples", "4",
    "--grid-size", "4",
], check=True)

In [ ]:
subprocess.run([
    sys.executable, "-u", str(runner / "run_t2_generation.py"),
    "--root", str(root),
    "--input", str(root / "data/secondary/instruction_tuning_synthetic/val.jsonl"),
    "--adapter", str(adapter),
    "--output", str(results / "t2_val_generation.jsonl"),
    "--max-examples", "4",
    "--max-new-tokens", "128",
], check=True)
subprocess.run([
    sys.executable, "-u", str(runner / "run_t2_generation.py"),
    "--root", str(root),
    "--input", str(root / "data/secondary/instruction_tuning_synthetic/hallucination_probe.jsonl"),
    "--adapter", str(adapter),
    "--output", str(results / "t2_probe_generation.jsonl"),
    "--max-examples", "8",
    "--max-new-tokens", "128",
], check=True)

In [ ]:
subprocess.run([
    sys.executable, "-u", str(runner / "attribute_t2_occlusion.py"),
    "--input", str(root / "data/secondary/instruction_tuning_synthetic/val.jsonl"),
    "--adapter", str(adapter),
    "--output", str(results / "t2_text_attributions.jsonl"),
    "--max-examples", "4",
    "--method", "shap",
    "--shap-samples", "32",
], check=True)

In [ ]:
import json

subprocess.run([
    sys.executable, str(runner / "evaluate_t2_generation.py"),
    "--input", str(results / "t2_val_generation.jsonl"),
    "--out", str(results / "t2_val_metrics.json"),
], check=True)
subprocess.run([
    sys.executable, str(runner / "evaluate_t2_generation.py"),
    "--input", str(results / "t2_probe_generation.jsonl"),
    "--out", str(results / "t2_probe_metrics.json"),
], check=True)
subprocess.run([
    sys.executable, str(runner / "evaluate_xai.py"),
    "--config", str(root / "configs/secondary.json"),
    "--image-input", str(results / "t1_image_attributions.jsonl"),
    "--text-input", str(results / "t2_text_attributions.jsonl"),
    "--out", str(results / "xai_metrics.json"),
], check=True)
for path in sorted(results.glob("*.json")):
    print(path.name, json.loads(path.read_text()))

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive("/content/xai-t2-results", "zip", root_dir=results.parent, base_dir=results.name)
print(archive)
files.download(archive)